# STRipy Pipeline Codebook
**Prostate Cancer AR CAG Repeat Genotyping**

This codebook documents the automated pipeline for genotyping androgen receptor (AR) CAG repeat expansions from whole-exome sequencing (WES) BAM files stored on the Seven Bridges Cancer Genomics Cloud (CGC).

---

## Pipeline Overview

```
CGC BAM Files → Download → samtools index → STRipy/ExpansionHunter → REViewer → Pileup → HTML Report → Delete BAM
```

**Tools Used:**
- [STRipy-pipeline](https://gitlab.com/andreassh/stripy-pipeline) — STR genotyping via Docker
- [ExpansionHunter](https://github.com/Illumina/ExpansionHunter) — repeat size estimation
- [REViewer](https://github.com/Illumina/REViewer) — read alignment visualization
- [samtools](http://www.htslib.org) — BAM indexing and pileup
- [Seven Bridges Python SDK](https://github.com/sbg/sevenbridges-python) — CGC API access
- [Plotly](https://plotly.com/python/) — interactive pileup visualization

## 1. Prerequisites & Setup

### Requirements
- Windows with WSL2 (Ubuntu)
- Docker Desktop (with WSL2 integration enabled)
- Python 3.12+
- samtools (installed in WSL)
- STRipy Docker image (`stripy:v2.5`)
- Reference genome: `hg38.fa` with `.fai` index
- Seven Bridges CGC account with API token

### File Structure
```
~/
├── stripy-pipeline/          # STRipy pipeline (cloned from GitLab)
│   ├── runDocker.sh
│   ├── config.json
│   └── stripy_results/       # Intermediate STRipy output
├── ref/
│   └── hg38.fa               # Reference genome (GRCh38)
├── bam/                      # Temporary BAM storage (deleted after each sample)
└── stripy_webapp/            # Flask web application
    ├── app.py
    └── templates/
        └── index.html
```

## 2. Installation

In [ ]:
# Install required Python packages (run in WSL)
# !pip install sevenbridges-python flask plotly matplotlib --break-system-packages

# Install samtools (run in WSL)
# !sudo apt-get install samtools -y

# Clone STRipy pipeline (run in WSL)
# !git clone https://gitlab.com/andreassh/stripy-pipeline.git ~/stripy-pipeline
# !chmod +x ~/stripy-pipeline/runDocker.sh

print('Prerequisites: Docker Desktop, WSL2 Ubuntu, Python 3.12+')

## 3. CGC API Connection

Connect to Seven Bridges Cancer Genomics Cloud using your API token.

> **Security Note:** Never hardcode your token. Use environment variables or enter it at runtime.

In [ ]:
import sevenbridges as sbg

# Connect to CGC
# Get token from: cgc.sbgenomics.com → your name → Developer → Authentication token
CGC_TOKEN = input('Enter your CGC API token: ')  # Never hardcode tokens
CGC_API_URL = 'https://cgc-api.sbgenomics.com/v2'

api = sbg.Api(url=CGC_API_URL, token=CGC_TOKEN)

# List accessible projects
projects = list(api.projects.query())
for p in projects:
    print(f'{p.id} - {p.name}')

## 4. Folder Navigation

Find folder IDs on CGC. Each folder has a unique ID used to query its contents.

In [ ]:
def get_all_files(api, project=None, parent=None):
    """
    Fetch all files from a CGC project or folder with pagination.
    
    Args:
        api: Seven Bridges API object
        project: Project ID string (e.g. 'username/project-name')
        parent: Folder ID string
    
    Returns:
        List of all file objects
    """
    all_files = []
    offset = 0
    limit = 100
    while True:
        if parent:
            batch = list(api.files.query(parent=parent, offset=offset, limit=limit))
        else:
            batch = list(api.files.query(project=project, offset=offset, limit=limit))
        all_files.extend(batch)
        if len(batch) < limit:
            break
        offset += limit
    return all_files


def list_folders(api, project_id):
    """
    List all folders and subfolders in a CGC project recursively.
    
    Args:
        api: Seven Bridges API object
        project_id: Project ID string
    """
    def recurse(parent=None, project=None, prefix=''):
        files = get_all_files(api, project=project, parent=parent)
        for f in files:
            if f.type == 'folder':
                display = f'{prefix}{f.name}'
                print(f'{display} --> ID: {f.id}')
                recurse(parent=f.id, prefix=f'  {prefix}')
    recurse(project=project_id)


# Example usage:
# list_folders(api, 'username/project-name')

## 5. BAM File Selection

Select the best BAM file per sample from a CGC folder. Priority order:
1. `.sorted.indexed.bam` (already indexed — fastest)
2. `.sorted.bam` (needs indexing)
3. `.bam` (unsorted — not recommended)

In [ ]:
def select_bam_files(files):
    """
    Select one BAM file per sample from a list of CGC files.
    Prefers sorted.indexed.bam > sorted.bam > .bam
    
    Args:
        files: List of CGC file objects
    
    Returns:
        List of selected BAM file objects (one per sample)
    """
    all_bams = [f for f in files if f.name.endswith('.bam') and f.type == 'file']
    indexed_bams = [f for f in all_bams if f.name.endswith('.sorted.indexed.bam')]
    sorted_bams = [f for f in all_bams if f.name.endswith('.sorted.bam')
                   and not f.name.endswith('.sorted.indexed.bam')]

    if indexed_bams:
        print(f'Using sorted.indexed.bam files ({len(indexed_bams)} found)')
        return sorted(indexed_bams, key=lambda f: f.name)
    elif sorted_bams:
        print(f'Using sorted.bam files ({len(sorted_bams)} found)')
        return sorted(sorted_bams, key=lambda f: f.name)
    else:
        print(f'Using all BAM files ({len(all_bams)} found)')
        return sorted(all_bams, key=lambda f: f.name)


# Example usage:
# FOLDER_ID = 'your_folder_id_here'
# folder = api.files.get(FOLDER_ID)
# files = get_all_files(api, parent=folder.id)
# bam_files = select_bam_files(files)
# for b in bam_files:
#     print(b.name)

## 6. Download BAM and BAI Files

Downloads BAM and matching BAI index file. Falls back to `samtools index` if no BAI exists.

In [ ]:
import subprocess
import os

BAM_DIR = '/home/ronak/bam'    # Temporary BAM storage in WSL

def download_and_index(bam, dest, all_files=None):
    """
    Download a BAM file and its index from CGC.
    Downloads .bai if available, otherwise runs samtools index.
    
    Args:
        bam: CGC file object for the BAM
        dest: Local destination path (e.g. '/home/ronak/bam/sample.bam')
        all_files: List of all files in the folder (to find matching .bai)
    
    Returns:
        True if successful, False if failed
    """
    print(f'Downloading {bam.name}...')
    bam.download(dest)

    # Try to download matching .bai file
    bai_downloaded = False
    if all_files:
        bai_name = bam.name + '.bai'
        bai = next((f for f in all_files if f.name == bai_name), None)
        if bai:
            print(f'Downloading index for {bam.name}...')
            bai.download(dest + '.bai')
            bai_downloaded = True

    # Fall back to samtools index if no .bai found
    if not bai_downloaded:
        print(f'No .bai found — indexing with samtools...')
        result = subprocess.run(['samtools', 'index', dest])
        if result.returncode != 0:
            print(f'WARNING: Indexing failed — file may be corrupted')
            return False

    return True

## 7. Run STRipy via Docker

STRipy runs inside a Docker container. The `runDocker.sh` script handles mounting paths and running ExpansionHunter + REViewer internally.

**Key parameters:**
- `-g`: Reference genome (`hg38`, `hg19`, or `hs1`)
- `-r`: Path to reference FASTA
- `-o`: Output folder
- `-l`: Locus/loci to genotype (e.g. `AR` or `AR,HTT,DMPK`)
- `-s`: Sample sex (`male` or `female`) — important for X-linked loci like AR
- `-i`: Input BAM file

In [ ]:
RESULTS_DIR = '/home/ronak/stripy-pipeline/stripy_results'
REF = '/home/ronak/ref/hg38.fa'
PIPELINE = '/home/ronak/stripy-pipeline/runDocker.sh'

def run_stripy(loci_str, genome='hg38', sex='male'):
    """
    Run STRipy pipeline via Docker on the current sample.bam.
    
    Args:
        loci_str: Comma-separated loci string (e.g. 'AR' or 'AR,HTT,DMPK')
        genome: Reference genome name ('hg38', 'hg19', 'hs1')
        sex: Sample sex ('male' or 'female')
              Use 'male' for prostate cancer samples (AR is X-linked)
    
    Output files (in RESULTS_DIR):
        sample.bam.html  — full STRipy report with read visualizations
        sample.bam.tsv   — genotyping results in tab-delimited format
        sample.bam.vcf   — variant call format output
    """
    print(f'Running STRipy (loci={loci_str}, genome={genome}, sex={sex})...')
    subprocess.run([
        'bash', PIPELINE,
        '-g', genome,
        '-r', REF,
        '-o', f'{RESULTS_DIR}/',
        '-l', loci_str,
        '-s', sex,
        '-i', f'{BAM_DIR}/sample.bam'
    ])


# STRipy config.json key settings:
# {
#   "num_threads": "all",          -- use all CPU cores
#   "output_html": true,           -- generate HTML report
#   "output_tsv": true,            -- generate TSV summary
#   "output_svg_flag_threshold": 0 -- generate read images for all loci
# }

## 8. REViewer — Read Alignment Visualization

REViewer creates read alignment images showing spanning reads (green) and flanking reads (blue) across the repeat locus. Only runs for flagged (non-normal) samples to save time.

In [ ]:
CATALOG = '/usr/local/bin/stripy-pipeline/catalog.json'  # Inside Docker container

def is_flagged():
    """
    Check if any locus in the TSV output is outside normal range.
    Used to decide whether to run REViewer (only for flagged samples).
    
    Returns:
        True if any locus is flagged (intermediate/pathogenic), False otherwise
    """
    tsv_path = f'{RESULTS_DIR}/sample.bam.tsv'
    if not os.path.exists(tsv_path):
        return False
    with open(tsv_path) as f:
        lines = f.readlines()
    if len(lines) < 2:
        return False
    headers = lines[0].strip().split('\t')
    for line in lines[1:]:
        values = line.strip().split('\t')
        row = dict(zip(headers, values))
        flag = row.get('Flag', '0')
        if flag and flag != '0' and flag != 'normal':
            return True
    return False


def run_reviewer(loci_str):
    """
    Run REViewer inside the STRipy Docker container.
    Generates SVG read alignment images for flagged loci.
    
    Args:
        loci_str: Comma-separated loci string
    
    Output:
        sample_reviewer.{LOCUS}.svg in RESULTS_DIR
    """
    subprocess.run([
        'docker', 'run', '--rm',
        '-v', f'{BAM_DIR}:/mnt/data',
        '-v', '/home/ronak/ref:/mnt/ref',
        '-v', f'{RESULTS_DIR}:/mnt/results',
        'stripy:v2.5', 'bash', '-c',
        f'REViewer '
        f'--reads /mnt/data/sample.bam '
        f'--vcf /mnt/results/sample.bam.vcf '
        f'--reference /mnt/ref/hg38.fa '
        f'--catalog {CATALOG} '
        f'--locus {loci_str} '
        f'--output-prefix /mnt/results/sample_reviewer'
    ])

## 9. Pileup Graph

Generates an interactive read depth pileup plot using samtools mpileup and Plotly.
Shows coverage across the AR CAG repeat locus (`chrX:67545316-67545395` on hg38).

In [ ]:
import plotly.graph_objects as go

# AR locus coordinates on hg38
AR_LOCUS = 'chrX:67545316-67545395'

def generate_pileup(sample_name):
    """
    Generate interactive read depth pileup for AR locus using samtools mpileup.
    
    Args:
        sample_name: Sample name for plot title
    
    Returns:
        Plotly HTML string for embedding in report, or empty string if no data
    """
    pileup_file = f'{RESULTS_DIR}/AR_pileup.txt'

    # Run samtools mpileup on AR locus
    subprocess.run([
        'samtools', 'mpileup',
        '-r', AR_LOCUS,
        '-f', REF,
        f'{BAM_DIR}/sample.bam'
    ], stdout=open(pileup_file, 'w'))

    # Parse pileup output
    positions, depths = [], []
    with open(pileup_file) as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 4:
                positions.append(int(parts[1]))
                depths.append(int(parts[3]))

    if not positions:
        return ''

    # Create interactive Plotly figure
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=positions,
        y=depths,
        fill='tozeroy',
        fillcolor='rgba(70, 130, 180, 0.4)',
        line=dict(color='steelblue'),
        hovertemplate='Position: %{x}<br>Depth: %{y}<extra></extra>'
    ))
    fig.update_layout(
        title=f'AR CAG Repeat Locus - Read Depth ({sample_name})',
        xaxis_title='Genomic Position (chrX)',
        yaxis_title='Read Depth',
        hovermode='x unified',
        template='plotly_white'
    )

    return fig.to_html(full_html=False, include_plotlyjs='cdn')

## 10. Report Generation

Merges STRipy HTML report, interactive pileup, and REViewer alignment into a single HTML file per sample.

In [ ]:
import shutil

OUTPUT_DIR = '/mnt/c/Users/YOUR_USERNAME/OneDrive/Desktop/ProstateProj/Stripy Results'

def generate_report(bam_name):
    """
    Generate a merged HTML report combining:
    - Genotyping summary table (from TSV)
    - STRipy full HTML report
    - Interactive Plotly pileup graph
    - REViewer SVG alignment image (if flagged)
    
    Args:
        bam_name: BAM filename used as report identifier
    
    Output:
        {bam_name}_full_report.html saved to OUTPUT_DIR
    """
    # Read STRipy HTML
    stripy_html = ''
    stripy_path = f'{RESULTS_DIR}/sample.bam.html'
    if os.path.exists(stripy_path):
        with open(stripy_path) as f:
            stripy_html = f.read()

    # Read REViewer SVG (only exists for flagged samples)
    reviewer_svg = ''
    reviewer_path = f'{RESULTS_DIR}/sample_reviewer.AR.svg'
    if os.path.exists(reviewer_path):
        with open(reviewer_path) as f:
            reviewer_svg = f.read()

    # Read TSV genotyping summary
    tsv_table = ''
    tsv_path = f'{RESULTS_DIR}/sample.bam.tsv'
    if os.path.exists(tsv_path):
        with open(tsv_path) as f:
            lines = f.readlines()
        if len(lines) >= 2:
            headers = lines[0].strip().split('\t')
            values = lines[1].strip().split('\t')
            tsv_table = '<table style="border-collapse:collapse;width:100%">'
            for h, v in zip(headers, values):
                tsv_table += (f'<tr>'
                              f'<td style="padding:6px;border:1px solid #ddd;font-weight:bold;background:#f5f5f5">{h}</td>'
                              f'<td style="padding:6px;border:1px solid #ddd">{v}</td>'
                              f'</tr>')
            tsv_table += '</table>'

    # Generate pileup
    pileup_html = generate_pileup(bam_name)

    # Build merged HTML
    merged_html = f'''<!DOCTYPE html>
<html>
<head>
    <title>STRipy Report - {bam_name}</title>
    <style>
        body {{ font-family: Arial, sans-serif; margin: 30px; background: #fafafa; }}
        h1 {{ color: #2c3e50; border-bottom: 3px solid #2980b9; padding-bottom: 10px; }}
        h2 {{ color: #2980b9; margin-top: 40px; }}
        .section {{ background: white; padding: 20px; margin-bottom: 30px;
                    border-radius: 8px; box-shadow: 0 1px 4px rgba(0,0,0,0.1); }}
    </style>
</head>
<body>
    <h1>STRipy Analysis Report</h1>
    <p style="color:#7f8c8d">Sample: {bam_name}</p>
    <div class="section"><h2>Genotyping Summary</h2>{tsv_table or "<p>No data</p>"}</div>
    <div class="section"><h2>STRipy Full Report</h2>{stripy_html}</div>
    <div class="section"><h2>Read Depth Pileup (Interactive)</h2>{pileup_html or "<p>No pileup data</p>"}</div>
    <div class="section"><h2>REViewer Read Alignment</h2>{reviewer_svg or "<p>No image (normal range)</p>"}</div>
</body>
</html>'''

    # Save to WSL results dir
    merged_path = f'{RESULTS_DIR}/{bam_name}_full_report.html'
    with open(merged_path, 'w') as f:
        f.write(merged_html)

    # Copy to Windows Desktop output folder
    try:
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        shutil.copy2(merged_path, f'{OUTPUT_DIR}/{bam_name}_full_report.html')
        print(f'Report saved: {bam_name}_full_report.html')
    except Exception as e:
        print(f'Warning: Could not copy to Desktop: {e}')

    # Cleanup temp files
    for temp in [stripy_path, reviewer_path, tsv_path,
                 f'{RESULTS_DIR}/AR_pileup.txt']:
        if os.path.exists(temp):
            os.remove(temp)

## 11. Full Batch Pipeline

Processes all BAM files in a CGC folder one at a time. Downloads next BAM in parallel while processing the current one to minimize total runtime.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def run_batch_pipeline(folder_id, loci='AR', genome='hg38', sex='male'):
    """
    Run STRipy pipeline on all BAM files in a CGC folder.
    
    Features:
    - Downloads .bai index files from CGC when available (faster than samtools index)
    - Parallel downloading: next BAM downloads while current is being analyzed
    - Skips REViewer for normal-range samples (saves time)
    - Deletes each BAM after processing (saves storage)
    - Saves merged HTML report per sample to OUTPUT_DIR
    
    Args:
        folder_id: CGC folder ID containing BAM files
        loci: Comma-separated loci to genotype (default: 'AR')
        genome: Reference genome ('hg38', 'hg19', 'hs1')
        sex: Sample sex ('male' or 'female')
              Important for X-linked loci — AR requires 'male' for prostate samples
    """
    # Connect and fetch files
    folder = api.files.get(folder_id)
    files = get_all_files(api, parent=folder.id)
    bam_files = select_bam_files(files)
    print(f'Found {len(bam_files)} BAM files')

    loci_str = loci if isinstance(loci, str) else ','.join(loci)

    with ThreadPoolExecutor(max_workers=1) as executor:
        next_future = None

        for i, bam in enumerate(bam_files):
            # Use pre-downloaded file or download first sample
            if next_future:
                success = next_future.result()
                if not success:
                    print(f'Skipping corrupted file: {bam.name}')
                    for f in [f'{BAM_DIR}/next.bam', f'{BAM_DIR}/next.bam.bai']:
                        if os.path.exists(f): os.remove(f)
                    continue
                os.rename(f'{BAM_DIR}/next.bam', f'{BAM_DIR}/sample.bam')
                if os.path.exists(f'{BAM_DIR}/next.bam.bai'):
                    os.rename(f'{BAM_DIR}/next.bam.bai', f'{BAM_DIR}/sample.bam.bai')
            else:
                success = download_and_index(bam, f'{BAM_DIR}/sample.bam', files)
                if not success:
                    print(f'Skipping {bam.name} — corrupted')
                    continue

            # Start pre-downloading next sample in background
            if i + 1 < len(bam_files):
                next_bam = bam_files[i + 1]
                for f in [f'{BAM_DIR}/next.bam', f'{BAM_DIR}/next.bam.bai']:
                    if os.path.exists(f): os.remove(f)
                next_future = executor.submit(
                    download_and_index, next_bam, f'{BAM_DIR}/next.bam', files)
            else:
                next_future = None

            # Run STRipy
            run_stripy(loci_str, genome, sex)

            # Run REViewer only if flagged
            if is_flagged():
                print(f'Sample flagged — running REViewer')
                run_reviewer(loci_str)
            else:
                print(f'Normal range — skipping REViewer')

            # Generate merged report
            generate_report(bam.name)

            # Cleanup BAM
            for f in [f'{BAM_DIR}/sample.bam', f'{BAM_DIR}/sample.bam.bai']:
                if os.path.exists(f): os.remove(f)

            print(f'Done with {bam.name}! ({i+1}/{len(bam_files)})')

    print('All samples complete!')


# Example usage:
# run_batch_pipeline(
#     folder_id='YOUR_FOLDER_ID',
#     loci='AR',
#     genome='hg38',
#     sex='male'
# )

## 12. Output Interpretation

### TSV Output Columns

| Column | Description |
|--------|-------------|
| `Locus` | Gene/locus name (e.g. AR) |
| `Coordinates` | Genomic coordinates (chrX:start-end) |
| `Motif` | Repeat unit (GCA for AR) |
| `Coverage` | Mean read depth at locus |
| `Allele1_Repeats` | Number of CAG repeats (allele 1) |
| `Allele2_Repeats` | Number of CAG repeats (allele 2) — NA for male samples |
| `Allele1_CI` | Confidence interval for allele 1 |
| `Allele1_IsOutlier` | True if allele is a population outlier (z-score ≥ 3.72) |
| `Allele1_DiseaseRange` | `normal`, `intermediate`, or `pathogenic` |
| `CorrespondingDiseases` | Associated disease (SBMA for AR) |
| `Filter` | `PASS` or reason for filter |

### AR CAG Repeat Ranges

| Range | Repeat Count | Classification |
|-------|-------------|----------------|
| Normal | 9–36 | No disease association |
| Intermediate | 37–38 | Uncertain significance |
| Pathogenic | ≥38 | Spinal and bulbar muscular atrophy (SBMA) |

### Note on Male vs Female Setting
AR is located on the X chromosome. Setting `-s male` tells STRipy to expect one allele (hemizygous). Setting `-s female` expects two alleles (diploid). Always use `male` for prostate cancer samples.

## 13. Flask Web Application

The pipeline is wrapped in a local Flask web app for easy use without command-line knowledge.

### Starting the App

In [ ]:
# To start the web app, run in WSL:
# cd ~/stripy_webapp
# python3 app.py
#
# Then open http://localhost:5000 in your browser
#
# Or double-click Start_STRipy.bat on your Windows Desktop

# Flask app routes:
# GET  /              — Main dashboard
# POST /run           — Start pipeline (params: token, folder_id, loci, genome, sex)
# GET  /logs          — Poll pipeline log messages
# GET  /progress      — Poll progress (current/total/percent)
# GET  /results       — List completed HTML reports
# GET  /download/<f>  — Download a report file
# POST /projects      — Fetch accessible CGC projects
# POST /folders       — Fetch folders in a project
# POST /refresh_folders — Re-fetch all folders live from CGC

print('Web app available at http://localhost:5000 when Flask is running')

## 14. Troubleshooting

| Problem | Cause | Fix |
|---------|-------|-----|
| `command 'docker' not found` | Docker Desktop not running | Open Docker Desktop and wait for it to load |
| `Pre-analysis check failed: Index for input file` | No .bai file | Run `samtools index sample.bam` or ensure .bai is in same folder |
| `Unauthorized` | Wrong or expired CGC token | Get new token from cgc.sbgenomics.com → Developer |
| `Destination folder is not found` | Wrong folder ID | Re-run folder listing to get correct ID |
| `EOF marker absent` / truncated BAM | Incomplete download | Re-download the BAM file |
| Genotyping Summary empty | STRipy failed silently | Check Docker is running; check TSV exists in results dir |
| Two alleles for male sample | `-s male` not applied | Ensure sex parameter is passed to `runDocker.sh` |
| Pipeline stops if laptop sleeps | WSL pauses on sleep | Set Windows power settings to Never sleep during batch runs |